# درس 09 - نمط تصميم ما وراء المعرفة


## الإعداد

يوضح دفتر الملاحظات هذا نمط تصميم ما وراء المعرفة باستخدام إطار عمل Microsoft Agent.

**المتطلبات المسبقة:**
- نشر Azure OpenAI مُكوَّن عبر متغيرات البيئة
- تم المصادقة على Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ما هو التفكير فوق المعرفي؟

التفكير فوق المعرفي هو **التفكير في التفكير**. في سياق وكلاء الذكاء الاصطناعي، يعني بناء وكلاء يمكنهم:

- **التأمل الذاتي** على مخرجاتهم وعملية استدلالهم
- **اكتشاف الأخطاء** والتعافي بسلاسة بدلاً من الفشل بصمت
- **تقييم** ما إذا كانت استجاباتهم كاملة ومفيدة
- **التكيف** استراتيجيتهم عندما لا تنجح المقاربة الأولى (على سبيل المثال، الرجوع إلى نظام احتياطي)

الوكيل فوق المعرفي لا يجيب على الأسئلة فحسب — بل يراقب أدائه ويعدل في الوقت الفعلي.


## الأدوات الأساسية والاحتياطية

نمط شائع في ما وراء المعرفة هو **استراتيجية الاحتياط**. يحاول الوكيل استخدام الأداة الأساسية أولاً؛ إذا فشلت (مثل خطأ 404)، يتعرّف الوكيل على الفشل وينتقل بشفافية إلى أداة احتياطية.

هذا يعكس الأنظمة الواقعية حيث قد تكون الخدمات الأساسية غير متاحة ويجب على الوكلاء تشخيص المشكلة بأنفسهم قبل اختيار مسار بديل.

فيما يلي نعرّف أداتين للبحث عن الرحلات:
- **الأساسية** — تغطي باريس وطوكيو وبرشلونة
- **الاحتياطية** — تغطي برلين وسيدني ومدينة نيويورك


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## وكيل ذاتي التأمل مع التعافي من الأخطاء

تم توجيه الوكيل أدناه لمحاولة نظام الطيران الأساسي أولاً، والتعرّف على حالات الفشل، والعودة بشفافية إلى نظام النسخ الاحتياطي. بعد كل استجابة يقوم بتقييم ذاتي موجز لمعرفة ما إذا كان قد أجاب بشكل كامل على سؤال المستخدم.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## نمط التقييم الذاتي

جانب آخر من ما وراء المعرفة هو **التقييم الذاتي**: وكيل منفصل (أو نفس الوكيل في مرور ثانٍ) يراجع الرد من حيث الاكتمال والدقة ومدى الفائدة.

فيما يلي ننشئ وكيل `ResponseEvaluator` يقوم بتقييم ردود وكيل السفر على ثلاثة أبعاد.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## الملخص

في هذا الدرس تعلمت كيفية بناء **وكلاء ما وراء المعرفة** باستخدام إطار عمل Microsoft Agent:

- **التأمل الذاتي**: وكلاء يراقبون استدلالهم الخاص ويتواصلون بشفافية حول ما حدث.
- **التعافي من الأخطاء باستخدام حلول بديلة**: نمط أداة أساسية + احتياطية حيث يكتشف الوكيل الإخفاقات (مثل أخطاء 404) ويحاول تلقائيًا استخدام مصدر بديل.
- **التقييم الذاتي**: وكيل تقييم منفصل يقيم الردود من حيث الاكتمال والدقة والفائدة.

تجعل هذه الأنماط الوكلاء أكثر متانة وشفافية وجديرة بالثقة — وهي صفات حاسمة لعمليات النشر في بيئات الإنتاج.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
إخلاء المسؤولية:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى جاهدين لتحقيق الدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر المعتمد. بالنسبة للمعلومات الحرجة، يُنصح بالاعتماد على ترجمة بشرية محترفة. لا نتحمل أي مسؤولية عن أي سوء فهم أو تفسير خاطئ ينشأ عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
